# Executable Examples of Constitutive Concerns in Quantum Software

This notebook presents the two executable examples associated with the WQSE 2026 paper **“Além da Ortogonalidade: Preocupações Constitutivas em Software Quântico”** (*Beyond Orthogonality: Constitutive Concerns in Quantum Software*):

1. backend adaptation: 16 logical operations become 13 native-basis operations after transpilation against `FakeFez`;
2. zero-noise extrapolation (ZNE): distribution fidelity changes from approximately 0.953 without mitigation to 0.977 after distribution-level extrapolation.

> **Scope.** The decorators shown in the paper are conceptual notation. They are not implemented by this artifact. Reusable domain logic is provided by the `quantum_constitutive_concerns` package; this notebook invokes its public command-line interface and presents the results.

## 1. Environment setup

When opened in Google Colab, the cell below clones the public repository. When executed from a local checkout, it locates and uses that checkout instead. The repository reference is explicit so that it can be changed from `main` to the camera-ready release tag after the release is created.

In [1]:
from pathlib import Path
import json
import math
import os
import subprocess
import sys
import tempfile

REPOSITORY_URL = "https://github.com/j3ffsilva/quantum-constitutive-concerns.git"
REPOSITORY_REF = "main"

def find_local_checkout():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate.resolve()
    return None

repository_root = find_local_checkout()
if repository_root is None:
    clone_root = Path("/content") if Path("/content").is_dir() else Path.cwd()
    repository_root = clone_root / "quantum-constitutive-concerns"
    if not repository_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
             REPOSITORY_URL, str(repository_root)],
            check=True,
        )

os.chdir(repository_root)
run_directory = Path(tempfile.mkdtemp(prefix="constitutive-concerns-"))
print("Repository checkout ready")
print("Temporary output directory created")

Repository checkout ready
Temporary output directory created


In [2]:
IN_COLAB = "google.colab" in sys.modules
local_python = repository_root / ".venv" / "bin" / "python"

if IN_COLAB:
    experiment_python = Path(sys.executable)
elif local_python.is_file():
    experiment_python = local_python
else:
    experiment_python = Path(sys.executable)

os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
subprocess.run(
    [str(experiment_python), "-m", "pip", "install",
     "--disable-pip-version-check", "--quiet", "-r",
     "requirements-lock.txt"],
    check=True,
)
subprocess.run(
    [str(experiment_python), "-m", "pip", "install",
     "--disable-pip-version-check", "--quiet", "--no-deps", "-e", "."],
    check=True,
)

os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning"

def run_experiment(command):
    completed = subprocess.run(command, capture_output=True, text=True)
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr, file=sys.stderr)
        completed.check_returncode()
    return completed

environment_name = "Colab runtime" if IN_COLAB else "local Python environment"
print(f"Experiment interpreter: {environment_name}")
print("Matplotlib backend fixed to Agg for non-interactive execution")

Experiment interpreter: local Python environment
Matplotlib backend fixed to Agg for non-interactive execution


The examples fix all parameters that affect the reported point estimates:

| Parameter | Value |
|---|---:|
| Backend snapshot | `FakeFez` |
| Marked state | `000` |
| Shots | 32,768 |
| Transpiler seed | 6 |
| Simulator seeds | 54,321–54,323 |
| Nominal folding factors | 1, 2, 3 |
| Operational threshold $\tau$ | 0.97 |

No live quantum hardware or IBM Quantum account is required.

## 2. Backend adaptation

In [3]:
backend_output = run_directory / "backend_adaptation.json"
run_experiment(
    [str(experiment_python), "-m", "quantum_constitutive_concerns",
     "backend-adaptation", "--output", str(backend_output)]
)
backend_result = json.loads(backend_output.read_text())
backend_expected = json.loads(Path("results/backend_adaptation.json").read_text())

assert backend_result["logical"]["size"] == 16
assert backend_result["native"]["size"] == 13
assert backend_result["native"]["operations"] == backend_expected["native"]["operations"]

print("Backend-adaptation example completed and verified")
print(f"Logical operations: {backend_result['logical']['size']}")
print(f"Native operations:  {backend_result['native']['size']}")
print(f"Native basis counts: {backend_result['native']['operations']}")

Backend-adaptation example completed and verified
Logical operations: 16
Native operations:  13
Native basis counts: {'cz': 2, 'rz': 5, 'sx': 6}


The result supports the structural claim in the paper: with `target=0`, optimization level 3, and transpiler seed 6, the 16-operation logical circuit is represented by 13 native-basis operations for the `FakeFez` target. These are transpiled instructions, not physical pulses, and the result is not a claim that every target or seed reduces the operation count.

## 3. ZNE error mitigation

For each nominal noise scale, the experiment samples a complete output distribution. It then linearly extrapolates each outcome probability to zero noise, clips possible negative values, normalizes the resulting vector, and computes classical distribution fidelity

$$F(P,Q)=\left(\sum_i\sqrt{P_iQ_i}\right)^2.$$

In [4]:
zne_output = run_directory / "zne_error_mitigation.json"
run_experiment(
    [str(experiment_python), "-m", "quantum_constitutive_concerns",
     "zne-error-mitigation", "--output", str(zne_output)]
)
zne_result = json.loads(zne_output.read_text())
zne_expected = json.loads(Path("results/zne_error_mitigation.json").read_text())

metrics = zne_result["metrics"]
expected_metrics = zne_expected["metrics"]
checked_metrics = [
    "noisy_distribution_fidelity",
    "zne_distribution_fidelity",
    "direct_linear_extrapolation_of_fidelity_diagnostic",
]
for name in checked_metrics:
    assert math.isclose(metrics[name], expected_metrics[name], abs_tol=5e-3), name

assert metrics["noisy_distribution_fidelity"] < 0.97
assert metrics["zne_distribution_fidelity"] >= 0.97

print("ZNE example completed and verified")
print(f"Fidelity without mitigation:       {metrics['noisy_distribution_fidelity']:.6f}")
print(f"Distribution-level ZNE fidelity:   {metrics['zne_distribution_fidelity']:.6f}")
print(f"Direct-fidelity diagnostic only:   {metrics['direct_linear_extrapolation_of_fidelity_diagnostic']:.6f}")
print(f"Native circuit sizes by scale:     {zne_result['circuits']['scaled_sizes']}")

ZNE example completed and verified
Fidelity without mitigation:       0.952961
Distribution-level ZNE fidelity:   0.977236
Direct-fidelity diagnostic only:   0.986777
Native circuit sizes by scale:     [191, 815, 881]


### Interpretation

- **0.953** is the classical fidelity between the noisy distribution at nominal scale 1 and the ideal distribution.
- **0.977** is the fidelity of the reconstructed, mitigated output distribution. This is the ZNE result used in the revised paper.
- **0.987** results from extrapolating the fidelity scalar itself. It is retained only as a diagnostic because computing it requires the ideal target distribution and it does not reconstruct an output distribution.

For the operational threshold adopted in the illustrative case, the result satisfies $0.953 < \tau=0.97 \leq 0.977$.

## 4. Automated checks

In [5]:
completed = subprocess.run(
    [str(experiment_python), "-m", "unittest", "discover", "-s",
     "tests", "-p", "test_*.py"],
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stderr or completed.stdout)
print("All artifact checks passed.")

...s.
----------------------------------------------------------------------
Ran 5 tests in 0.243s

OK (skipped=1)

All artifact checks passed.


## 5. Limitations

- The experiment uses a noise model derived from the `FakeFez` calibration snapshot; it is not a live-hardware execution.
- The reported fidelities are conditioned on the pinned package versions, noise model, seeds, and number of shots.
- Nominal folding factors 1, 2, and 3 produce 191, 815, and 881 native-basis instructions after inverse gates are translated back to the Fez basis; these are not physical-pulse counts or exact scale multiples.
- The threshold $\tau=0.97$ is an operational criterion for this illustrative instance, not a universal threshold.
- The ideal distribution is available because this is a benchmark; production criteria may instead require partial properties, analytical bounds, calibration, or dynamic evidence.
- The artifact executes two illustrative examples; it does not constitute empirical validation of the proposed taxonomy across algorithms or hardware platforms.

## 6. Software versions

In [6]:
print(json.dumps(zne_result["versions"], indent=2, sort_keys=True))

{
  "mitiq": "1.0.0",
  "qiskit": "2.4.2",
  "qiskit_aer": "0.17.2",
  "qiskit_ibm_runtime": "0.47.0"
}
